# 🏷 Сегментация текста: находим людей, организации и места в новостях

> Сначала: **Файл → Сохранить копию на Диске**. Затем: **Среда выполнения → Сменить среду → GPU** (без GPU тоже работает, но медленнее).

**Сегментация картинки** даёт класс каждому пикселю. **Сегментация текста** даёт класс каждому слову:

| Слово | Медведев | посетил | завод | КамАЗ | в | Татарстане |
|---|---|---|---|---|---|---|
| Тег | B-PER | O | O | B-ORG | O | B-LOC |

Сегодня проходим весь путь настоящего ML-проекта: **EDA → предобработка → разделение → модель → гиперпараметры → проверка качества**.

Каждое задание: замените `None` на свой код → **Shift + Enter** → запустите ячейку 🧪 под ним.

In [ ]:
#@title ⚙️ Запустите первой: библиотеки, данные, проверки { display-mode: "form" }
import os, time, urllib.request
from collections import Counter
from itertools import product
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from IPython.display import HTML, display

if not os.path.exists("ner_news.csv"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Scr1ptG/ml_for_3p1/main/07-segment/data/ner_news.csv", "ner_news.csv")
torch.manual_seed(42); np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

def check(tests, n):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}\n   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание {n} выполнено!")
    return True


COLORS = {"PER": "#ffd8d8", "ORG": "#dbe9ff", "LOC": "#dff7ea"}
def show_tags(words, tags):
    """Рисует предложение: люди красным, организации синим, места зелёным."""
    html = ""
    for w, t in zip(words, tags):
        c = COLORS.get(t[2:]) if t != "O" else None
        html += f"<span style='background:{c};padding:2px 4px;border-radius:4px;color:#111'>{w} <b style='font-size:10px'>{t}</b></span> " if c else w + " "
    display(HTML(f"<p style='font-size:16px;line-height:2'>{html}</p>"))

print("✅ Готово. Устройство:", device)

---
# Часть 1. EDA: смотрим на данные

In [ ]:
df = pd.read_csv("ner_news.csv")
sentences = [s.split(" ") for s in df["tokens"]]
tags = [t.split(" ") for t in df["tags"]]
print(df.head(3).to_string())
show_tags(sentences[0], tags[0])
show_tags(sentences[7], tags[7])

**Схема тегов BIO:** `B-` — первое слово сущности (**B**egin), `I-` — продолжение (**I**nside), `O` — всё остальное (**O**utside). Благодаря этому «Дмитрий Медведев» — одна сущность: `B-PER I-PER`. Набор: Collection3, русские новости.

### ✏️ Задание 1. Сколько предложений `n_sent` и сколько всего слов `n_words`?

In [ ]:
n_sent = None
n_words = None

In [ ]:
#@title 🧪 Проверка задания 1 { display-mode: "form" }
if check([
    ('предложений', lambda: n_sent == len(df), 'Сколько элементов в списке sentences?'),
    ('слов', lambda: n_words == sum(len(x) for x in sentences), 'Нужно сложить длины всех предложений'),
], 1):
    print(f"{n_sent} предложений, {n_words} слов")

<details><summary>💡 Подсказка</summary>

`sum` умеет складывать значения из генератора.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
n_sent = ___(sentences)
n_words = sum(___ for s in sentences)
```
</details>

### ✏️ Задание 2. Посчитайте теги: `tag_counts` (Counter) и долю тега O `share_o`

In [ ]:
tag_counts = Counter()
for t in tags:
    pass  # обновите счётчик

share_o = None

In [ ]:
#@title 🧪 Проверка задания 2 { display-mode: "form" }
if check([
    ('тегов 7 видов', lambda: len(tag_counts) == 7, 'Добавьте в счётчик теги каждого предложения'),
    ('доля O', lambda: 0.8 < share_o < 0.9, 'Число O разделите на число всех тегов'),
], 2):
    pd.Series(tag_counts).sort_values().plot.barh(title="Сколько каких тегов"); plt.show()
    print(f"Тег O: {share_o:.1%}. Запомните это число!")

<details><summary>💡 Подсказка</summary>

У Counter есть метод `update`, а `values()` даёт все счётчики.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
tag_counts.___(t)
share_o = tag_counts["O"] / sum(___)
```
</details>

### ✏️ Задание 3. Длины предложений: `lengths` и `MAX_LEN` — длина, в которую укладываются 95% предложений

In [ ]:
lengths = None
MAX_LEN = None

In [ ]:
#@title 🧪 Проверка задания 3 { display-mode: "form" }
if check([
    ('список длин', lambda: len(lengths) == n_sent, 'Длина для каждого предложения'),
    ('95-й перцентиль', lambda: MAX_LEN == int(np.percentile(lengths, 95)), 'В numpy есть функция percentile'),
], 3):
    plt.hist(lengths, bins=50); plt.axvline(MAX_LEN, color="r"); plt.title(f"MAX_LEN = {MAX_LEN}"); plt.show()
    print("Сеть принимает батч одной длины: длинные обрежем, короткие дополним.")

<details><summary>💡 Подсказка</summary>

Посмотрите `np.percentile(a, q)`. Результат приведите к int.
</details>

---
# Часть 2. Разделение и предобработка

**Важно:** сначала делим, потом строим словарь. Иначе данные проверки «протекут» в обучение.

### ✏️ Задание 4. Разделите данные: 80% train, 10% val, 10% test

In [ ]:
idx = np.arange(n_sent)
train_idx, rest_idx = None, None
val_idx, test_idx = None, None

In [ ]:
#@title 🧪 Проверка задания 4 { display-mode: "form" }
if check([
    ('train около 80%', lambda: abs(len(train_idx) / n_sent - 0.8) < 0.01, 'test_size=0.2 в первом разбиении'),
    ('val и test поровну', lambda: abs(len(val_idx) - len(test_idx)) <= 1, 'Остаток делим пополам'),
    ('не пересекаются', lambda: not set(train_idx) & set(test_idx) and not set(val_idx) & set(test_idx), 'Второй раз делите rest_idx'),
], 4):
    print(len(train_idx), len(val_idx), len(test_idx))
    print("train учит, val выбирает гиперпараметры, test — честная оценка один раз в конце.")

<details><summary>💡 Подсказка</summary>

`train_test_split` вызывается дважды.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
train_idx, rest_idx = train_test_split(idx, test_size=___, random_state=42)
val_idx, test_idx = train_test_split(___, test_size=___, random_state=42)
```
</details>

### ✏️ Задание 5. Постройте словарь `word2idx` только по train: слова в нижнем регистре, встретившиеся минимум 2 раза

In [ ]:
PAD, UNK = 0, 1
counts = Counter(w.lower() for i in train_idx for w in sentences[i])
word2idx = {"<PAD>": PAD, "<UNK>": UNK}
for w, c in counts.items():
    pass  # если c >= 2, дайте слову следующий номер

In [ ]:
#@title 🧪 Проверка задания 5 { display-mode: "form" }
if check([
    ('служебные токены', lambda: word2idx["<PAD>"] == 0 and word2idx["<UNK>"] == 1, 'Не меняйте первые два элемента'),
    ('только частые слова', lambda: len(word2idx) == 2 + sum(c >= 2 for c in counts.values()), 'Добавляйте слово, только если c >= 2'),
    ('номера без дыр', lambda: sorted(word2idx.values()) == list(range(len(word2idx))), 'Следующий номер равен текущей длине словаря'),
], 5):
    print("Размер словаря:", len(word2idx), "из", len(counts))

<details><summary>💡 Подсказка</summary>

Какое значение всегда свободно в растущем словаре? Его текущая длина.
</details>

In [ ]:
tag2idx = {t: i for i, t in enumerate(sorted(tag_counts))}
idx2tag = {i: t for t, i in tag2idx.items()}
print(tag2idx)

### ✏️ Задание 6. Функция `encode(words)`: слова → номера. Незнакомое слово → UNK, регистр нижний

In [ ]:
def encode(words):
    return None

In [ ]:
#@title 🧪 Проверка задания 6 { display-mode: "form" }
if check([
    ('знакомое слово', lambda: encode(["и"]) == [word2idx["и"]], 'Ищите слово в word2idx'),
    ('регистр', lambda: encode(["И"]) == [word2idx["и"]], 'Сначала приведите к нижнему регистру'),
    ('незнакомое', lambda: encode(["ъъъъъ"]) == [UNK], 'У словаря есть метод get со значением по умолчанию'),
], 6):
    print(sentences[0][:6], "→", encode(sentences[0][:6]))

<details><summary>💡 Подсказка</summary>

`словарь.get(ключ, запасное_значение)`.
</details>

### ✏️ Задание 7. Функция `pad(seq, value)`: обрезать до MAX_LEN или дополнить значением value

In [ ]:
def pad(seq, value):
    seq = seq[:MAX_LEN]
    return None

In [ ]:
#@title 🧪 Проверка задания 7 { display-mode: "form" }
if check([
    ('короткая', lambda: pad([5], 0) == [5] + [0] * (MAX_LEN - 1), 'Сколько значений не хватает до MAX_LEN?'),
    ('длинная', lambda: len(pad([1] * 500, 0)) == MAX_LEN, 'Длина должна быть ровно MAX_LEN'),
], 7):
    pass

<details><summary>💡 Подсказка</summary>

Список можно умножить на число.
</details>

In [ ]:
def to_tensors(ids):
    X = torch.tensor([pad(encode(sentences[i]), PAD) for i in ids])
    # -100: на дополненных позициях сеть не штрафуем
    Y = torch.tensor([pad([tag2idx[t] for t in tags[i]], -100) for i in ids])
    return X, Y

X_train, Y_train = to_tensors(train_idx)
X_val, Y_val = to_tensors(val_idx)
X_test, Y_test = to_tensors(test_idx)
print(X_train.shape, Y_train.shape)

---
# Часть 3. Модель

Для картинки свёртка смотрит на соседние пиксели, для текста — на **соседние слова**. «завод **КамАЗ** в» подсказывает организацию, поэтому берём `Conv1d`. Рекуррентные сети будут в следующей теме.

```
номера слов  (батч, длина)
Embedding →  (батч, длина, emb_dim)
Conv1d x2 →  (батч, filters, длина)    окно из kernel слов
Linear    →  (батч, длина, 7 тегов)    ответ для КАЖДОГО слова
```

### ✏️ Задание 8. Соберите `WordTagger`: заполните слои по схеме

In [ ]:
class WordTagger(nn.Module):
    def __init__(self, vocab, n_tags, emb_dim, filters, kernel, dropout):
        super().__init__()
        self.emb = None  # nn.Embedding: сколько слов, длина вектора, padding_idx=PAD
        self.conv1 = nn.Conv1d(emb_dim, filters, kernel, padding=kernel // 2)
        self.conv2 = None  # такая же свёртка, но вход уже filters
        self.drop = nn.Dropout(dropout)
        self.head = None  # nn.Linear: filters -> n_tags

    def forward(self, x):
        e = self.drop(self.emb(x)).transpose(1, 2)   # Conv1d ждёт (батч, каналы, длина)
        h = torch.relu(self.conv2(torch.relu(self.conv1(e))))
        return self.head(self.drop(h.transpose(1, 2)))

In [ ]:
#@title 🧪 Проверка задания 8 { display-mode: "form" }
if check([
    ('форма выхода', lambda: WordTagger(100, 7, 8, 8, 3, 0.1)(torch.zeros(2, 20).long()).shape == (2, 20, 7), 'Выход: батч × длина × число тегов'),
    ('паддинг', lambda: WordTagger(100, 7, 8, 8, 3, 0.1).emb.padding_idx == 0, 'Передайте padding_idx=PAD'),
], 8):
    print(WordTagger(len(word2idx), 7, 64, 64, 3, 0.2))

<details><summary>💡 Подсказка</summary>

Посмотрите на схему: что входит в слой и что выходит.
</details>

---
# Часть 4. Метрика: почему accuracy здесь врёт

Модель, которая на **все** слова отвечает O, не находит ничего, но accuracy у неё высокая.

### ✏️ Задание 9. Посчитайте accuracy такой «ленивой» модели на val: `lazy_acc`

In [ ]:
mask = Y_val != -100
lazy_acc = None

In [ ]:
#@title 🧪 Проверка задания 9 { display-mode: "form" }
if check([
    ('значение', lambda: 0.75 < lazy_acc < 0.95, 'Какая доля настоящих тегов равна номеру O?'),
], 9):
    print(f"Ничего не найдя, модель получает accuracy {lazy_acc:.1%}! Поэтому смотрим на F1.")

<details><summary>💡 Подсказка</summary>

Сравнение тензора с числом даёт True/False. Приведите к float и возьмите среднее.
</details>

**Precision** — сколько из найденного правильно. **Recall** — сколько из существующего нашли. **F1** — их среднее гармоническое, высокое только если хороши оба.

### ✏️ Задание 10. Модель нашла 8 сущностей, 6 верны. Всего в тексте их 12. Посчитайте `precision`, `recall`, `f1`

In [ ]:
precision = None
recall = None
f1 = None

In [ ]:
#@title 🧪 Проверка задания 10 { display-mode: "form" }
if check([
    ('precision', lambda: precision == 0.75, 'Верные / найденные'),
    ('recall', lambda: recall == 0.5, 'Верные / существующие'),
    ('f1', lambda: abs(f1 - 0.6) < 1e-9, '2·P·R / (P + R)'),
], 10):
    pass

In [ ]:
#@title ⚙️ Запустите: обучение и оценка (читать не обязательно) { display-mode: "form" }
ALL_TAG_IDS = [i for t, i in tag2idx.items() if t != "O"]

def predict_ids(model, X, batch=256):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            out.append(model(X[i:i+batch].to(device)).argmax(-1).cpu())
    return torch.cat(out)

def evaluate(model, X, Y):
    """Возвращает (accuracy, F1). F1 считается по всем тегам, кроме O."""
    p, mask = predict_ids(model, X), Y != -100
    t, p = Y[mask].numpy(), p[mask].numpy()
    return (t == p).mean(), f1_score(t, p, labels=ALL_TAG_IDS, average="macro", zero_division=0)

def train_model(params, verbose=True):
    torch.manual_seed(42)
    model = WordTagger(len(word2idx), len(tag2idx), params["emb_dim"], params["filters"],
                       params["kernel"], params["dropout"]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=params["lr"])
    loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
    loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=params["batch"], shuffle=True)
    hist, best, best_state = [], -1, None
    for ep in range(1, params["epochs"] + 1):
        model.train(); total = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = loss_fn(model(xb).reshape(-1, len(tag2idx)), yb.reshape(-1))
            opt.zero_grad(); loss.backward(); opt.step(); total += loss.item() * len(xb)
        acc, f1 = evaluate(model, X_val, Y_val)
        hist.append((total / len(X_train), acc, f1))
        if f1 > best:
            best, best_state = f1, {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if verbose:
            print(f"эпоха {ep}: loss {hist[-1][0]:.3f} | val accuracy {acc:.3f} | val F1 {f1:.3f}")
    model.load_state_dict(best_state)
    return model, hist, best

print("✅ Готово")

---
# Часть 5. Гиперпараметры

**Параметры** (веса) сеть учит сама. **Гиперпараметры** выбираем мы, до обучения:

| Что | Мало | Много |
|---|---|---|
| `lr` шаг обучения | учится долго | скачет, не сходится |
| `emb_dim` длина вектора слова | мало смысла | переобучение, медленно |
| `filters` число фильтров | не хватает сил | переобучение |
| `kernel` окно свёртки | не видит соседей | размывает |
| `dropout` | переобучение | недообучение |
| `batch`, `epochs` | шумно / не доучилась | медленно / переобучилась |

Как подбирать: **ручной перебор → сетка (grid search) → случайный поиск → Optuna**. Сравнивать только по **val**.

### ✏️ Задание 11. Задайте первые гиперпараметры и обучите модель (3 эпохи, lr 0.001)

In [ ]:
params = {"emb_dim": 64, "filters": 64, "kernel": 3, "dropout": 0.2, "batch": 64,
          "lr": None, "epochs": None}
model, hist, best_f1 = train_model(params)

In [ ]:
#@title 🧪 Проверка задания 11 { display-mode: "form" }
if check([
    ('lr', lambda: params["lr"] == 0.001, 'Запишите 0.001'),
    ('обучение прошло', lambda: best_f1 > 0.3, 'F1 должен быть заметно выше нуля; проверьте модель'),
], 11):
    print(f"Лучший val F1: {best_f1:.3f}, а у ленивой модели F1 = 0")

<details><summary>💡 Подсказка</summary>

Вписываем только два числа.
</details>

### ✏️ Задание 12. Сделайте перебор по сетке: `lr` и `kernel`, результаты в `results`

In [ ]:
grid = {"lr": [0.0003, 0.003], "kernel": [3, 5]}
results = []
for lr, kernel in product(grid["lr"], grid["kernel"]):
    p = {**params, "lr": lr, "kernel": kernel, "epochs": 2}
    _, _, f1 = None, None, 0  # обучите с параметрами p без печати (verbose=False)
    results.append({"lr": lr, "kernel": kernel, "val_f1": f1})
    print(results[-1])

table = pd.DataFrame(results).sort_values("val_f1", ascending=False)
best_row = None  # первая строка таблицы

In [ ]:
#@title 🧪 Проверка задания 12 { display-mode: "form" }
if check([
    ('4 запуска', lambda: len(results) == 4, 'Сетка 2 × 2'),
    ('обучение идёт', lambda: max(r["val_f1"] for r in results) > 0.2, 'Вызовите train_model(p, verbose=False)'),
    ('лучшая строка', lambda: best_row["val_f1"] == max(r["val_f1"] for r in results), 'Таблица уже отсортирована, возьмите iloc[0]'),
], 12):
    display(table)

<details><summary>💡 Подсказка</summary>

Перебор занимает пару минут. Подумайте: сколько запусков было бы при 5 гиперпараметрах по 3 значения?
</details>

### ✏️ Задание 13. Обучите финальную модель с лучшими `lr` и `kernel` (5 эпох)

In [ ]:
final_params = {**params, "lr": None, "kernel": None, "epochs": 5}
final, hist, best_f1 = train_model(final_params)

In [ ]:
#@title 🧪 Проверка задания 13 { display-mode: "form" }
if check([
    ('лучшие значения', lambda: final_params["lr"] == best_row["lr"] and final_params["kernel"] == best_row["kernel"], 'Возьмите значения из best_row'),
    ('качество', lambda: best_f1 > 0.5, 'Если F1 низкий, проверьте, что lr взят из best_row'),
], 13):
    h = np.array(hist)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3)); ax[0].plot(h[:, 0]); ax[0].set_title("loss"); ax[1].plot(h[:, 1], label="accuracy"); ax[1].plot(h[:, 2], label="F1"); ax[1].legend(); plt.show()

<details><summary>💡 Подсказка</summary>

`kernel` должен быть целым: `int(...)`.
</details>

---
# Часть 6. Честная проверка на test

### ✏️ Задание 14. Посчитайте `test_acc` и `test_f1` функцией `evaluate`

In [ ]:
test_acc, test_f1 = None, None

In [ ]:
#@title 🧪 Проверка задания 14 { display-mode: "form" }
if check([
    ('F1', lambda: test_f1 > 0.5, 'evaluate(модель, X, Y) с test'),
    ('accuracy выше ленивой', lambda: test_acc > lazy_acc, 'Передайте final, X_test, Y_test'),
], 14):
    print(f"accuracy {test_acc:.3f}, F1 {test_f1:.3f}")
    p, m = predict_ids(final, X_test), Y_test != -100
    print(classification_report(Y_test[m].numpy(), p[m].numpy(), labels=ALL_TAG_IDS, target_names=[idx2tag[i] for i in ALL_TAG_IDS], zero_division=0))

<details><summary>💡 Подсказка</summary>

Функция возвращает два числа.
</details>

### ✏️ Задание 15. Разметьте своё предложение: `my_text` и функция `tag_text`

In [ ]:
my_text = None  # напишите новость: человек, компания, город

def tag_text(text):
    words = text.split()
    x = torch.tensor([pad(encode(words), PAD)]).to(device)
    final.eval()
    with torch.no_grad():
        ids = None  # argmax по последней оси, строка [0]
    return [idx2tag[int(i)] for i in ids[:len(words)]]

In [ ]:
#@title 🧪 Проверка задания 15 { display-mode: "form" }
if check([
    ('свой текст', lambda: isinstance(my_text, str) and len(my_text.split()) >= 4, 'Минимум 4 слова'),
    ('тег на слово', lambda: len(tag_text(my_text)) == len(my_text.split()), 'Сколько слов, столько тегов'),
], 15):
    show_tags(my_text.split(), tag_text(my_text))

<details><summary>💡 Подсказка</summary>

Вызовите модель на x и возьмите самый вероятный класс для каждого слова.
</details>

---
## 🏁 Итог
EDA → разделение → словарь (только train) → паддинг → Conv1d → F1 вместо accuracy → перебор по val → один замер на test.

**Как улучшить:** предобученные эмбеддинги, LSTM (следующая тема), BERT, оценка целых сущностей через `seqeval`, готовые инструменты: `spaCy`, `Natasha`, `DeepPavlov`.